# Data Quality

This notebook validates the E-SMAT Silver datasets before the analytical layer is created.

## Responsibilities

1. Load reusable functions;
2. Read the Silver Delta tables;
3. Validate required schemas;
4. Check mandatory business keys;
5. Detect duplicate records;
6. Validate accepted values and numeric ranges;
7. Evaluate header-detail referential integrity;
8. Verify requirement enrichment and site consistency;
9. Validate conditional fields;
10. Consolidate quality results and failure records;
11. Persist data-quality monitoring tables;
12. Stop the pipeline when blocking checks fail.

## Outputs

- `data_quality_summary`
- `data_quality_failures`

A quality check is classified as:

- **error**: blocks the downstream analytical load;
- **warning**: requires review but does not stop the pipeline;
- **info**: provides monitoring context.

> Production catalog and schema names are represented by placeholders in this portfolio version.


## 1. Load shared functions

In [ ]:
%run ./01_shared_functions


## 2. Configure Silver sources and quality targets

In [ ]:
SILVER_CATALOG = "<catalog>"
SILVER_SCHEMA = "<silver-schema>"
QUALITY_CATALOG = "<catalog>"
QUALITY_SCHEMA = "<quality-schema>"

SILVER_TABLES = {
    "inspection_header": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_header_silver"
    ),
    "inspection_responses": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_responses_silver"
    ),
    "inspection_requirements": (
        f"{SILVER_CATALOG}.{SILVER_SCHEMA}.inspection_requirements_silver"
    ),
}

QUALITY_TABLES = {
    "summary": (
        f"{QUALITY_CATALOG}.{QUALITY_SCHEMA}.data_quality_summary"
    ),
    "failures": (
        f"{QUALITY_CATALOG}.{QUALITY_SCHEMA}.data_quality_failures"
    ),
}

WRITE_MODE = "overwrite"
OVERWRITE_SCHEMA = "true"
FAIL_ON_ERROR = True


## 3. Validate configuration and read Silver tables

In [ ]:
def validate_quality_config(source_tables, target_tables):
    required_sources = {
        "inspection_header",
        "inspection_responses",
        "inspection_requirements",
    }
    missing_sources = sorted(required_sources - set(source_tables))

    if missing_sources:
        raise ValueError(
            f"Missing Silver source configuration: {missing_sources}"
        )

    for table_group, table_mapping in (
        ("source", source_tables),
        ("target", target_tables),
    ):
        for table_name, qualified_name in table_mapping.items():
            if not qualified_name or qualified_name.count(".") != 2:
                raise ValueError(
                    f"Invalid {table_group} table '{table_name}': "
                    f"{qualified_name}"
                )

    print("Data-quality configuration validated successfully.")


validate_quality_config(SILVER_TABLES, QUALITY_TABLES)

silver_dataframes = {}

for dataset_name, table_name in SILVER_TABLES.items():
    if not spark.catalog.tableExists(table_name):
        raise RuntimeError(
            f"Silver table not found for '{dataset_name}': {table_name}"
        )

    dataframe = spark.table(table_name)

    if len(dataframe.columns) != len(set(dataframe.columns)):
        raise ValueError(
            f"Silver dataset '{dataset_name}' contains duplicate columns."
        )

    silver_dataframes[dataset_name] = dataframe
    print(f"Silver dataset loaded: {dataset_name}")


## 4. Validate required Silver schemas

In [ ]:
REQUIRED_COLUMNS = {
    "inspection_header": {
        "inspection_id",
        "site",
        "inspection_date",
        "inspection_year",
        "shift",
        "activity_type",
        "contract_type",
        "unsafe_condition",
        "unsafe_condition_description",
        "safety_tag_opened",
        "safety_tag_number",
        "near_miss",
        "near_miss_description",
        "audit_type",
        "header_key",
        "processing_timestamp",
    },
    "inspection_responses": {
        "inspection_id",
        "site",
        "question_id",
        "safe_act",
        "unsafe_act",
        "inspection_date",
        "audit_type",
        "questionnaire_version",
        "theme",
        "category_id",
        "response_key",
        "processing_timestamp",
    },
    "inspection_requirements": {
        "requirement_key",
        "site",
        "methodology",
        "questionnaire_version",
        "theme",
        "category_id",
        "question_id",
        "question_number",
        "active",
        "processing_timestamp",
    },
}

for dataset_name, required_columns in REQUIRED_COLUMNS.items():
    require_columns(
        dataframe=silver_dataframes[dataset_name],
        required_columns=required_columns,
        dataset_name=f"{dataset_name}_silver",
    )

print("Required Silver schemas validated successfully.")


## 5. Quality-check framework

Each check produces one summary row and zero or more detailed failure rows. Failure details use a generic structure so results from different datasets can be stored together.


In [ ]:
quality_results = []
quality_failure_dataframes = []


def register_quality_check(
    check_id,
    dataset_name,
    check_name,
    severity,
    failure_df,
    key_columns,
    description,
):
    if severity not in {"error", "warning", "info"}:
        raise ValueError(
            f"Invalid severity for check '{check_id}': {severity}"
        )

    require_columns(
        dataframe=failure_df,
        required_columns=set(key_columns),
        dataset_name=f"quality_check_{check_id}",
    )

    failure_count = failure_df.count()
    status = "passed" if failure_count == 0 else "failed"

    quality_results.append(
        {
            "check_id": check_id,
            "dataset": dataset_name,
            "check_name": check_name,
            "severity": severity,
            "status": status,
            "failure_count": failure_count,
            "description": description,
        }
    )

    if failure_count > 0:
        key_expression = F.concat_ws(
            " | ",
            *[
                F.concat(
                    F.lit(f"{column_name}="),
                    F.coalesce(
                        F.col(column_name).cast("string"),
                        F.lit("<null>"),
                    ),
                )
                for column_name in key_columns
            ],
        )

        standardized_failures = failure_df.select(
            F.lit(check_id).alias("check_id"),
            F.lit(dataset_name).alias("dataset"),
            F.lit(check_name).alias("check_name"),
            F.lit(severity).alias("severity"),
            key_expression.alias("record_key"),
            F.lit(description).alias("failure_reason"),
        )

        quality_failure_dataframes.append(standardized_failures)

    print(
        f"{check_id} | {dataset_name} | {status} | "
        f"failures={failure_count}"
    )


## 6. Mandatory-key checks

In [ ]:
header_df = silver_dataframes["inspection_header"]
responses_df = silver_dataframes["inspection_responses"]
requirements_df = silver_dataframes["inspection_requirements"]

register_quality_check(
    check_id="DQ001",
    dataset_name="inspection_header",
    check_name="mandatory_header_keys",
    severity="error",
    failure_df=header_df.filter(
        F.col("inspection_id").isNull()
        | (F.trim(F.col("inspection_id")) == "")
        | F.col("inspection_date").isNull()
        | F.col("site").isNull()
        | (F.trim(F.col("site")) == "")
    ),
    key_columns=["inspection_id", "site"],
    description=(
        "Inspection header must contain inspection_id, site, and inspection_date."
    ),
)

register_quality_check(
    check_id="DQ002",
    dataset_name="inspection_responses",
    check_name="mandatory_response_keys",
    severity="error",
    failure_df=responses_df.filter(
        F.col("inspection_id").isNull()
        | (F.trim(F.col("inspection_id")) == "")
        | F.col("question_id").isNull()
        | (F.trim(F.col("question_id")) == "")
        | F.col("response_key").isNull()
        | (F.trim(F.col("response_key")) == "")
    ),
    key_columns=["inspection_id", "question_id", "response_key"],
    description=(
        "Question-level response must contain inspection_id, question_id, and response_key."
    ),
)

register_quality_check(
    check_id="DQ003",
    dataset_name="inspection_requirements",
    check_name="mandatory_requirement_keys",
    severity="error",
    failure_df=requirements_df.filter(
        F.col("requirement_key").isNull()
        | (F.trim(F.col("requirement_key")) == "")
        | F.col("question_id").isNull()
        | (F.trim(F.col("question_id")) == "")
        | F.col("site").isNull()
        | (F.trim(F.col("site")) == "")
    ),
    key_columns=["requirement_key", "question_id", "site"],
    description=(
        "Requirement must contain requirement_key, question_id, and site."
    ),
)


## 7. Duplicate checks

In [ ]:
duplicate_headers_df = (
    header_df.groupBy("inspection_id")
    .count()
    .filter(F.col("count") > 1)
)

register_quality_check(
    check_id="DQ004",
    dataset_name="inspection_header",
    check_name="unique_inspection_id",
    severity="error",
    failure_df=duplicate_headers_df,
    key_columns=["inspection_id", "count"],
    description="inspection_id must be unique in the header dataset.",
)

duplicate_responses_df = (
    responses_df.groupBy("response_key")
    .count()
    .filter(F.col("count") > 1)
)

register_quality_check(
    check_id="DQ005",
    dataset_name="inspection_responses",
    check_name="unique_response_key",
    severity="error",
    failure_df=duplicate_responses_df,
    key_columns=["response_key", "count"],
    description="response_key must be unique in the response dataset.",
)

duplicate_requirements_df = (
    requirements_df.groupBy("requirement_key")
    .count()
    .filter(F.col("count") > 1)
)

register_quality_check(
    check_id="DQ006",
    dataset_name="inspection_requirements",
    check_name="unique_requirement_key",
    severity="error",
    failure_df=duplicate_requirements_df,
    key_columns=["requirement_key", "count"],
    description="requirement_key must be unique in the requirements dataset.",
)


## 8. Accepted-value and range checks

In [ ]:
register_quality_check(
    check_id="DQ007",
    dataset_name="inspection_header",
    check_name="accepted_audit_type",
    severity="error",
    failure_df=header_df.filter(
        F.col("audit_type").isNull()
        | ~F.col("audit_type").isin("EMAT", "SMAT", "ESMAT")
    ),
    key_columns=["inspection_id", "audit_type"],
    description="audit_type must be EMAT, SMAT, or ESMAT.",
)

register_quality_check(
    check_id="DQ008",
    dataset_name="inspection_requirements",
    check_name="accepted_theme",
    severity="error",
    failure_df=requirements_df.filter(
        F.col("theme").isNull()
        | ~F.col("theme").isin("environment", "safety")
    ),
    key_columns=["requirement_key", "theme"],
    description="Requirement theme must be environment or safety.",
)

register_quality_check(
    check_id="DQ009",
    dataset_name="inspection_responses",
    check_name="accepted_response_theme",
    severity="error",
    failure_df=responses_df.filter(
        F.col("theme").isNull()
        | ~F.col("theme").isin("environment", "safety")
    ),
    key_columns=["response_key", "theme"],
    description=(
        "Enriched response theme must be environment or safety."
    ),
)

register_quality_check(
    check_id="DQ010",
    dataset_name="inspection_responses",
    check_name="valid_act_quantity_range",
    severity="error",
    failure_df=responses_df.filter(
        F.col("safe_act").isNull()
        | F.col("unsafe_act").isNull()
        | ~F.col("safe_act").between(0, 3)
        | ~F.col("unsafe_act").between(0, 3)
    ),
    key_columns=["response_key", "safe_act", "unsafe_act"],
    description="safe_act and unsafe_act must contain values from 0 to 3.",
)

register_quality_check(
    check_id="DQ011",
    dataset_name="inspection_responses",
    check_name="at_least_one_observation",
    severity="warning",
    failure_df=responses_df.filter(
        (F.coalesce(F.col("safe_act"), F.lit(0)) == 0)
        & (F.coalesce(F.col("unsafe_act"), F.lit(0)) == 0)
    ),
    key_columns=["response_key", "safe_act", "unsafe_act"],
    description=(
        "Response contains neither a safe nor an unsafe observation."
    ),
)


## 9. Header-detail referential integrity

In [ ]:
header_keys_df = header_df.select("inspection_id").dropDuplicates()

orphan_responses_df = responses_df.join(
    header_keys_df,
    on="inspection_id",
    how="left_anti",
)

register_quality_check(
    check_id="DQ012",
    dataset_name="inspection_responses",
    check_name="response_has_header",
    severity="error",
    failure_df=orphan_responses_df,
    key_columns=["inspection_id", "response_key"],
    description=(
        "Every question-level response must reference an existing inspection header."
    ),
)

response_header_context_df = responses_df.alias("response").join(
    header_df.select(
        "inspection_id",
        F.col("site").alias("header_site"),
        F.col("inspection_date").alias("header_date"),
        F.col("audit_type").alias("header_audit_type"),
    ).alias("header"),
    on="inspection_id",
    how="inner",
)

inconsistent_header_context_df = response_header_context_df.filter(
    (F.col("response.site") != F.col("header_site"))
    | (F.col("response.inspection_date") != F.col("header_date"))
    | (F.col("response.audit_type") != F.col("header_audit_type"))
)

register_quality_check(
    check_id="DQ013",
    dataset_name="inspection_responses",
    check_name="response_header_context_consistency",
    severity="error",
    failure_df=inconsistent_header_context_df,
    key_columns=["inspection_id", "response_key"],
    description=(
        "Response site, inspection date, and audit type must match the header context."
    ),
)


## 10. Requirement enrichment and referential integrity

In [ ]:
active_requirement_keys_df = (
    requirements_df
    .filter(F.col("active") == F.lit(True))
    .select(
        F.col("site").alias("requirement_site"),
        F.col("methodology").alias("requirement_methodology"),
        F.col("question_id").alias("requirement_question_id"),
    )
    .dropDuplicates()
)

responses_without_requirement_df = responses_df.alias("response").join(
    active_requirement_keys_df.alias("requirement"),
    (
        F.col("response.site") == F.col("requirement.requirement_site")
    )
    & (
        F.col("response.audit_type")
        == F.col("requirement.requirement_methodology")
    )
    & (
        F.col("response.question_id")
        == F.col("requirement.requirement_question_id")
    ),
    how="left_anti",
)

register_quality_check(
    check_id="DQ014",
    dataset_name="inspection_responses",
    check_name="response_has_active_requirement",
    severity="error",
    failure_df=responses_without_requirement_df,
    key_columns=["response_key", "site", "audit_type", "question_id"],
    description=(
        "Every response must reference an active requirement for the same site and methodology."
    ),
)

missing_enrichment_df = responses_df.filter(
    F.col("questionnaire_version").isNull()
    | F.col("theme").isNull()
    | F.col("category_id").isNull()
    | F.col("question_number").isNull()
)

register_quality_check(
    check_id="DQ015",
    dataset_name="inspection_responses",
    check_name="complete_requirement_enrichment",
    severity="error",
    failure_df=missing_enrichment_df,
    key_columns=["response_key", "question_id"],
    description=(
        "Response enrichment must include questionnaire version, theme, category, and question number."
    ),
)


## 11. Conditional-field checks

In [ ]:
unsafe_condition_inconsistency_df = header_df.filter(
    (
        (F.col("unsafe_condition") == F.lit(True))
        & (
            F.col("unsafe_condition_description").isNull()
            | (F.trim(F.col("unsafe_condition_description")) == "")
        )
    )
    | (
        (F.col("unsafe_condition") == F.lit(False))
        & F.col("unsafe_condition_description").isNotNull()
        & (F.trim(F.col("unsafe_condition_description")) != "")
    )
)

register_quality_check(
    check_id="DQ016",
    dataset_name="inspection_header",
    check_name="unsafe_condition_description_consistency",
    severity="error",
    failure_df=unsafe_condition_inconsistency_df,
    key_columns=["inspection_id", "unsafe_condition"],
    description=(
        "Unsafe-condition description must be populated only when unsafe_condition is true."
    ),
)

tag_inconsistency_df = header_df.filter(
    (
        (F.col("safety_tag_opened") == F.lit(True))
        & (
            F.col("safety_tag_number").isNull()
            | (F.trim(F.col("safety_tag_number")) == "")
        )
    )
    | (
        (F.col("safety_tag_opened") == F.lit(False))
        & F.col("safety_tag_number").isNotNull()
        & (F.trim(F.col("safety_tag_number")) != "")
    )
    | (
        (F.col("safety_tag_opened") == F.lit(True))
        & (F.col("unsafe_condition") != F.lit(True))
    )
)

register_quality_check(
    check_id="DQ017",
    dataset_name="inspection_header",
    check_name="safety_tag_consistency",
    severity="error",
    failure_df=tag_inconsistency_df,
    key_columns=["inspection_id", "safety_tag_opened", "safety_tag_number"],
    description=(
        "Tag number must be populated only when a tag is opened, and an opened tag requires an unsafe condition."
    ),
)

near_miss_inconsistency_df = header_df.filter(
    (
        (F.col("near_miss") == F.lit(True))
        & (
            F.col("near_miss_description").isNull()
            | (F.trim(F.col("near_miss_description")) == "")
        )
    )
    | (
        (F.col("near_miss") == F.lit(False))
        & F.col("near_miss_description").isNotNull()
        & (F.trim(F.col("near_miss_description")) != "")
    )
)

register_quality_check(
    check_id="DQ018",
    dataset_name="inspection_header",
    check_name="near_miss_description_consistency",
    severity="error",
    failure_df=near_miss_inconsistency_df,
    key_columns=["inspection_id", "near_miss"],
    description=(
        "Near-miss description must be populated only when near_miss is true."
    ),
)


## 12. Date and questionnaire checks

In [ ]:
register_quality_check(
    check_id="DQ019",
    dataset_name="inspection_header",
    check_name="valid_inspection_date",
    severity="error",
    failure_df=header_df.filter(
        F.col("inspection_date").isNull()
        | (F.col("inspection_year") != F.year(F.col("inspection_date")))
    ),
    key_columns=["inspection_id", "inspection_date", "inspection_year"],
    description=(
        "inspection_date must be valid and inspection_year must match it."
    ),
)

invalid_question_number_df = requirements_df.filter(
    F.col("question_number").isNull()
    | (F.col("question_number") <= 0)
)

register_quality_check(
    check_id="DQ020",
    dataset_name="inspection_requirements",
    check_name="valid_question_number",
    severity="error",
    failure_df=invalid_question_number_df,
    key_columns=["requirement_key", "question_number"],
    description="question_number must be a positive integer.",
)

active_requirement_duplicates_df = (
    requirements_df
    .filter(F.col("active") == F.lit(True))
    .groupBy(
        "site",
        "methodology",
        "questionnaire_version",
        "question_id",
    )
    .count()
    .filter(F.col("count") > 1)
)

register_quality_check(
    check_id="DQ021",
    dataset_name="inspection_requirements",
    check_name="unique_active_requirement_context",
    severity="error",
    failure_df=active_requirement_duplicates_df,
    key_columns=[
        "site",
        "methodology",
        "questionnaire_version",
        "question_id",
        "count",
    ],
    description=(
        "An active question must be unique within site, methodology, and questionnaire version."
    ),
)


## 13. Monitoring checks

Monitoring checks provide useful context without blocking the pipeline.


In [ ]:
headers_without_responses_df = header_df.join(
    responses_df.select("inspection_id").dropDuplicates(),
    on="inspection_id",
    how="left_anti",
)

register_quality_check(
    check_id="DQ022",
    dataset_name="inspection_header",
    check_name="header_has_responses",
    severity="warning",
    failure_df=headers_without_responses_df,
    key_columns=["inspection_id", "site", "audit_type"],
    description="Submitted inspection header has no question-level responses.",
)

unsafe_without_description_df = responses_df.filter(
    (F.col("unsafe_act") > 0)
    & (
        F.col("description").isNull()
        | (F.trim(F.col("description")) == "")
    )
)

register_quality_check(
    check_id="DQ023",
    dataset_name="inspection_responses",
    check_name="unsafe_observation_has_description",
    severity="warning",
    failure_df=unsafe_without_description_df,
    key_columns=["response_key", "unsafe_act"],
    description=(
        "Response with an unsafe observation should contain a description."
    ),
)


## 14. Consolidate and display quality results

In [ ]:
quality_summary_df = add_processing_timestamp(
    spark.createDataFrame(quality_results)
).withColumn(
    "pipeline_layer",
    F.lit("silver"),
)

if quality_failure_dataframes:
    quality_failures_df = quality_failure_dataframes[0]

    for failure_dataframe in quality_failure_dataframes[1:]:
        quality_failures_df = quality_failures_df.unionByName(
            failure_dataframe
        )

    quality_failures_df = add_processing_timestamp(
        quality_failures_df
    ).withColumn(
        "pipeline_layer",
        F.lit("silver"),
    )
else:
    failure_schema = StructType(
        [
            StructField("check_id", StringType(), False),
            StructField("dataset", StringType(), False),
            StructField("check_name", StringType(), False),
            StructField("severity", StringType(), False),
            StructField("record_key", StringType(), True),
            StructField("failure_reason", StringType(), False),
        ]
    )

    quality_failures_df = add_processing_timestamp(
        spark.createDataFrame([], schema=failure_schema)
    ).withColumn(
        "pipeline_layer",
        F.lit("silver"),
    )

quality_overview_df = (
    quality_summary_df
    .groupBy("severity", "status")
    .agg(
        F.count("check_id").alias("check_count"),
        F.sum("failure_count").alias("failure_count"),
    )
    .orderBy("severity", "status")
)

display(quality_summary_df.orderBy("check_id"))
display(quality_overview_df)

if quality_failures_df.limit(1).count() > 0:
    display(quality_failures_df.orderBy("check_id", "record_key"))
else:
    print("No detailed quality failures were generated.")


## 15. Persist quality-monitoring tables

In [ ]:
for dataset_name, dataframe in {
    "summary": quality_summary_df,
    "failures": quality_failures_df,
}.items():
    target_table = QUALITY_TABLES[dataset_name]

    print(f"Writing quality dataset '{dataset_name}' to {target_table}...")

    (
        dataframe.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", OVERWRITE_SCHEMA)
        .saveAsTable(target_table)
    )

    print(f"Quality table written successfully: {target_table}")


## 16. Apply the pipeline quality gate

Only failed checks with severity `error` block the downstream analytical load. Warnings remain available for monitoring and review.


In [ ]:
blocking_failures = [
    result
    for result in quality_results
    if result["severity"] == "error"
    and result["status"] == "failed"
]

warning_failures = [
    result
    for result in quality_results
    if result["severity"] == "warning"
    and result["status"] == "failed"
]

print(
    f"Quality checks completed: {len(quality_results)} checks, "
    f"{len(blocking_failures)} blocking failures, "
    f"{len(warning_failures)} warnings."
)

if FAIL_ON_ERROR and blocking_failures:
    failed_check_ids = [
        result["check_id"]
        for result in blocking_failures
    ]

    raise RuntimeError(
        "Silver data-quality gate failed. "
        f"Blocking checks: {failed_check_ids}"
    )

print("Silver data-quality gate passed successfully.")


## Next step

Run `05_delta_load.ipynb` after the quality gate passes. The next notebook creates the analytical classification table and publishes validated datasets for Power BI consumption.
